# MLB Team Tracker — Interactive Demo

This notebook walks through every module of the sox-tracker suite.
By default it uses the **Boston Red Sox (2025)** — change `TEAM_ABBR` and `SEASON` below to follow any team.

**Prerequisites:** run `python fetch.py --team BOS --season 2025` before executing this notebook.

In [ ]:
import sys
sys.path.insert(0, '..')  # make sure repo root is on the path

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import plotly.io as pio
pio.renderers.default = 'notebook'

# ── Configure your team here ─────────────────────────────────────────────
TEAM_ABBR = 'BOS'
SEASON    = 2025
# ─────────────────────────────────────────────────────────────────────────

from config import TEAMS
TEAM_INFO = TEAMS[TEAM_ABBR]
TEAM_ID   = TEAM_INFO['id']
TEAM_NAME = TEAM_INFO['name']

print(f'Team : {TEAM_NAME}')
print(f'Season: {SEASON}')

## 1. Load cached data

In [ ]:
from data.fetcher import Fetcher

fetcher  = Fetcher(team_id=TEAM_ID, season=SEASON)
games    = fetcher.load('games')
batting  = fetcher.load('batting')
pitching = fetcher.load('pitching')
fielding = fetcher.load('fielding')
roster   = fetcher.load('roster')

print(f'Games   : {len(games)}')
print(f'Batting : {len(batting)} rows  ({batting["player_id"].nunique()} players)')
print(f'Pitching: {len(pitching)} rows  ({pitching["player_id"].nunique()} pitchers)')
print(f'Fielding: {len(fielding)} rows')
print(f'Roster  : {len(roster)} players')

## 2. Season Overview

In [ ]:
from analysis.standings import season_record, opponent_splits, pace_projection

rec    = season_record(games)
splits = opponent_splits(games, TEAM_ID)
proj_w, proj_l = pace_projection(rec['wins'], rec['losses'])

print(f"Record : {rec['wins']}-{rec['losses']}  ({rec['win_pct']:.3f})")
print(f"Run Diff: {rec['run_diff']:+d}")
print(f"Pythagorean: {rec['pyth_wins']}-{rec['pyth_losses']}")
print(f"162-Game Pace: {proj_w}-{proj_l}")
print()
print(f"Home : {rec['home_w']}-{rec['home_l']}   Away: {rec['away_w']}-{rec['away_l']}")
print(f"vs Div: {splits['div_w']}-{splits['div_l']}   vs Lg: {splits['lg_w']}-{splits['lg_l']}   Inter: {splits['inter_w']}-{splits['inter_l']}")
print(f"Last 7: {rec['last_7_w']}-{rec['last_7_l']}   Last 15: {rec['last_15_w']}-{rec['last_15_l']}")

## 3. Season Timeline Chart

In [ ]:
from viz.charts import season_timeline
season_timeline(games, TEAM_NAME).show()

In [ ]:
from viz.charts import rolling_win_pct_chart
rolling_win_pct_chart(games, TEAM_NAME, windows=[7, 15]).show()

In [ ]:
from viz.charts import run_differential_chart
run_differential_chart(games, TEAM_NAME).show()

## 4. Offense

In [ ]:
from analysis.offense import team_offense_summary, batting_leaderboard

off = team_offense_summary(batting, games)
print(f"AVG/OBP/SLG/OPS : {off['avg']:.3f}/{off['obp']:.3f}/{off['slg']:.3f}/{off['ops']:.3f}")
print(f"BABIP: {off['babip']:.3f}   HR: {off['hr']}   SB: {off['sb']}")
print(f"BB%: {off['bb_pct']:.1f}%   K%: {off['k_pct']:.1f}%   R/G: {off['r_per_g']:.2f}")

lb = batting_leaderboard(batting, min_pa=50)
lb[['player_name','pa','avg','obp','slg','ops','hr','rbi','sb','babip','bb_pct','k_pct']].head(12)

In [ ]:
from viz.charts import batting_leaderboard_heatmap
from analysis.offense import player_season_totals

totals = player_season_totals(batting)
batting_leaderboard_heatmap(totals, TEAM_NAME).show()

In [ ]:
from analysis.offense import hot_cold_summary
from viz.charts import hot_cold_chart

hc = hot_cold_summary(batting)
display(hc[['player_name','season_ops','last_7_ops','last_15_ops','delta','label']].head(15))
hot_cold_chart(hc, TEAM_NAME).show()

In [ ]:
from analysis.offense import lineup_slot_production

lineup_slot_production(batting)[['batting_order','pa','avg','obp','slg','ops','hr','rbi','bb_pct','k_pct']]

## 5. Streaks

In [ ]:
from analysis.streaks import (
    current_streak, longest_streak, streak_timeline,
    series_results, series_summary, all_hitting_streaks, monthly_record
)
from viz.charts import streak_timeline_chart

stype, slen = current_streak(games)
print(f'Current streak : {stype}{slen}')
print(f'Longest W streak: {longest_streak(games, "W")}')
print(f'Longest L streak: {longest_streak(games, "L")}')

st = streak_timeline(games)
streak_timeline_chart(st, TEAM_NAME).show()

In [ ]:
ss = series_summary(games)
print(f"Series: {ss['sweeps']} sweeps / {ss['splits']} splits / {ss['series_losses']} series losses")
print(f"Series win rate: {ss['series_win_pct']}%")

series_results(games).tail(10)

In [ ]:
monthly_record(games)

## 6. Pitching

In [ ]:
from analysis.pitching import (
    team_pitching_split, starter_season_totals, bullpen_season_totals,
    bullpen_role_splits, quality_start_correlation, ace_correlation,
    rotation_rest_tracker, bullpen_overuse_alerts
)

split = team_pitching_split(pitching)
print(f"ERA: {split['era']}   WHIP: {split['whip']}   K/9: {split['k_per_9']}   BB/9: {split['bb_per_9']}")
print(f"SP ERA: {split['starter_era']}   RP ERA: {split['bullpen_era']}   QS%: {split['quality_start_pct']}%")
print(f"SV: {split['saves']}   HLD: {split['holds']}   BS: {split['blown_saves']}")

In [ ]:
sp = starter_season_totals(pitching)
sp[['player_name','gs','w','l','ip_str','era','fip','whip','k_per_9','bb_per_9','p_per_ip','avg_game_score','quality_start_pct']]

In [ ]:
from viz.charts import rotation_heatmap, era_split_chart
rotation_heatmap(pitching, TEAM_NAME).show()

In [ ]:
qs = quality_start_correlation(pitching, games)
print(f"Quality Start record: {qs['qs_w']}-{qs['qs_l']} ({qs['qs_win_pct']:.3f})")
print(f"Non-QS record: {qs['non_qs_w']}-{qs['non_qs_l']} ({qs['non_qs_win_pct']:.3f})")

ace_correlation(pitching, games)

In [ ]:
bullpen_role_splits(pitching)

In [ ]:
from viz.charts import bullpen_load_chart
bullpen_load_chart(pitching, TEAM_NAME).show()

## 7. Defense

In [ ]:
from analysis.defense import team_fielding_summary, player_fielding_totals, catcher_stats

summary = team_fielding_summary(fielding, games)
print(f"FLD%: {summary['fielding_pct']:.4f}   Errors: {summary['errors']}   E/G: {summary['errors_per_game']}   DP: {summary['dp']}")

player_fielding_totals(fielding).head(15)

In [ ]:
catcher_stats(fielding)

## 8. Historical Trends

In [ ]:
from analysis.history import fetch_season_records, records_within_reach, NOTABLE_SEASONS

historical = fetch_season_records(TEAM_ID, start_year=2000)
historical.tail(10)

In [ ]:
from viz.charts import multi_season_win_pct

notable = NOTABLE_SEASONS.get(TEAM_ABBR, {})
multi_season_win_pct(historical, TEAM_NAME, highlight_seasons=notable).show()

In [ ]:
records = records_within_reach(games, historical, TEAM_ABBR)
for r in records:
    print(f"{r['record']:35s}  Current: {r['current']}   Best: {r['franchise_best']} ({r['holder_season']})")

## 9. Build the full interactive dashboard

In [ ]:
from viz.dashboard import build

path = build(
    games=games, batting=batting, pitching=pitching, fielding=fielding,
    team_name=TEAM_NAME, team_abbr=TEAM_ABBR, season=SEASON,
)
print(f'Dashboard written → {path}')
print('Open in your browser to explore all charts interactively.')